<a href="https://colab.research.google.com/github/tomheston/fragility-metrics/blob/main/notebooks/modality_compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# Fragility Metrics Toolkit: Modality Comparison (two diagnostic tests, same reference standard)
# modality_compare.py  v.09-OCT-2026-r1
#
# Compares the sensitivity, specificity, accuracy, PPV and NPV of two
# imaging modalities (or any two diagnostic tests) and reports, for each
# comparison, the complete statistical evidence triplet p-fr-nb as
# defined in FRAGILITY_METRICS.md v14.4.1.
#
# Input per modality: TP, FN, FP, TN against the same reference standard.
# Optional per metric: the discordant-pair counts b and c when the two
# modalities read the SAME cases (paired design), so that an exact
# McNemar comparison can replace the unpaired comparison.
#
# Comparison design and metrics
# -----------------------------
#   UNPAIRED (default; the only option when a paper reports marginals only):
#       table {correct_A, wrong_A, correct_B, wrong_B} on the relevant subset
#       p  : two-sided Fisher's exact
#       fr : GFQ = GFI / N  (GFI certified exact by the toolkit engine)
#       nb : RQ = |ad - bc| / (N^2 / 4)
#       Reported with the flag "unpaired analysis" because when both
#       modalities read the same cases the true design is paired and the
#       unpaired test ignores the within-case correlation.
#   PAIRED (when b and c are supplied for that metric):
#       b = cases correct on A and wrong on B; c = wrong on A, correct on B
#       p  : exact McNemar (binomial test of b against b + c at 0.5, two-sided)
#       fr : PFI-M (McNemar-path percent fragility index, decimal scale)
#       nb : MHQ = |b - c| / (b + c)
#
# Interpretation bands:
#   fr band (GFQ only): unstable when GFI = 1 or GFQ < 0.01, else stable.
#   RQ is reported as a value only. The canon near/intermediate/far cutoffs
#   were derived from independent-sample trial tables and are not applied
#   to modality-comparison tables here. PFI-M and MHQ have no canon bands.
#
# Input modes
# -----------
#   Mode 1 (counts):     TP, FN, FP, TN per modality.
#   Mode 3 (accuracy):   correct and n per modality for ONE metric (for
#       papers that report only an accuracy or correct-staging count).
#       Unpaired route only: Fisher exact, GFQ, RQ on {correct1, wrong1,
#       correct2, wrong2}.
#   Mode 2 (published):  n_pos, n_neg, sensitivity %, specificity % per
#       modality, as papers usually report them. TP and TN are back-solved
#       from the percentages. Because a rounded percentage can be consistent
#       with more than one integer count, every count that reproduces the
#       reported percentage at its stated precision is enumerated and the
#       comparison metrics are reported as a range (identical bounds = firm).
#       A percentage that no integer count reproduces is flagged. This is
#       arithmetic on published aggregates, not patient-level reconstruction,
#       and tables obtained this way are labeled "reconstructed".
#
# GFI/RQ engine: imports fragility_metrics_r12.py if it sits beside this
# file; otherwise downloads fragility_metrics_r11.py from the public
# GitHub repository. Certified GFI values are identical between r11 and r12.
#
# Google Colab: paste this whole file into one cell and run it, or
#   !wget -q <raw url of this file> && python modality_compare.py
# then answer the prompts. Programmatic use:
#   res = compare_modalities(ct=(23,10,112,248), pet=(28,5,58,302),
#                            names=("CT","PET/CT"))
#
# IF YOU USE THIS CALCULATOR PLEASE CITE:
# Heston, T. F. (2025). Fragility Metrics Toolkit [Software]. Zenodo.
# https://doi.org/10.5281/zenodo.17254763
#
# (c) Thomas F. Heston 2025-2026. CC-BY 4.0

import importlib
import os
import sys
import urllib.request

try:
    import scipy  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])

from scipy.stats import beta, binomtest, chi2, fisher_exact

ALPHA = 0.05

TOOLKIT_LOCAL = "fragility_metrics_r12"
TOOLKIT_FALLBACK_URL = (
    "https://raw.githubusercontent.com/tomheston/fragility-metrics/"
    "main/notebooks/fragility_metrics_r11.py"
)


def _load_toolkit():
    """Import the 2x2 toolkit (local r12 preferred, GitHub r11 fallback)."""
    sys.path.insert(0, os.getcwd())
    try:
        return importlib.import_module(TOOLKIT_LOCAL), TOOLKIT_LOCAL
    except ImportError:
        pass
    fname = "fragility_metrics_r11.py"
    if not os.path.isfile(fname):
        urllib.request.urlretrieve(TOOLKIT_FALLBACK_URL, fname)
    return importlib.import_module("fragility_metrics_r11"), "fragility_metrics_r11"


FM, TOOLKIT_NAME = _load_toolkit()


# ---------- per-modality metrics ----------

def clopper_pearson(k, n, alpha=ALPHA):
    if n == 0:
        return (None, None)
    lo = 0.0 if k == 0 else float(beta.ppf(alpha / 2, k, n - k + 1))
    hi = 1.0 if k == n else float(beta.ppf(1 - alpha / 2, k + 1, n - k))
    return lo, hi


def metric_counts(tp, fn, fp, tn):
    """(correct, n_relevant) for each diagnostic metric, per canon 3.4."""
    return {
        "sensitivity": (tp, tp + fn),
        "specificity": (tn, tn + fp),
        "ppv": (tp, tp + fp),
        "npv": (tn, tn + fn),
        "accuracy": (tp + tn, tp + fn + fp + tn),
    }


# ---------- unpaired comparison: Fisher + GFQ + RQ ----------

def unpaired_triplet(kA, nA, kB, nB, alpha=ALPHA):
    a, b, c, d = kA, nA - kA, kB, nB - kB
    N = a + b + c + d
    p = float(fisher_exact([[a, b], [c, d]], alternative="two-sided")[1])
    gfi_res = FM.compute_gfi_gfq(a, b, c, d, alpha=alpha, certify=True)
    rq = FM.compute_ufi_pfi(a, b, c, d, alpha=alpha)["RQ"]
    gfi = gfi_res.get("GFI")
    gfq = gfi_res.get("GFQ")
    if gfi is None:
        fr_band = "not computed"
    elif gfi == 1 or gfq < 0.01:
        fr_band = "unstable"
    else:
        fr_band = "stable"
    return {
        "design": "unpaired analysis (Fisher exact; GFQ + RQ)",
        "table": (a, b, c, d), "N": N,
        "p": p, "significant": p < alpha,
        "GFI": gfi, "GFQ": gfq, "GFI_certified": bool(gfi_res.get("certified")),
        "GFI_note": gfi_res.get("note", ""),
        "fr_band": fr_band, "RQ": rq,
    }


# ---------- mode 2: back-solve counts from published percentages ----------

def _decimals(s):
    s = str(s).strip().rstrip("%")
    return len(s.split(".")[1]) if "." in s else 0


def candidate_counts(pct, n):
    """
    All integer k in 0..n whose percentage 100*k/n rounds to the reported
    value at the reported precision. `pct` may be a string ("84.8") so that
    the stated precision is preserved; a float is taken at its own precision.
    """
    nd = _decimals(pct)
    target = round(float(str(pct).strip().rstrip("%")), nd)
    return [k for k in range(n + 1) if round(100.0 * k / n, nd) == target]


def reconstruct_tables(n_pos, n_neg, sens_pct, spec_pct):
    """
    Returns (tables, note). `tables` is the list of (TP, FN, FP, TN)
    consistent with the published sensitivity and specificity; empty if a
    percentage cannot be reproduced by any integer count.
    """
    tps = candidate_counts(sens_pct, n_pos)
    tns = candidate_counts(spec_pct, n_neg)
    notes = []
    if not tps:
        notes.append(f"sensitivity {sens_pct}% is not reproducible with n_pos = {n_pos}")
    if not tns:
        notes.append(f"specificity {spec_pct}% is not reproducible with n_neg = {n_neg}")
    tables = [(tp, n_pos - tp, n_neg - tn, tn) for tp in tps for tn in tns]
    if len(tps) > 1:
        notes.append(f"TP ambiguous at stated precision: {tps}")
    if len(tns) > 1:
        notes.append(f"TN ambiguous at stated precision: {tns}")
    return tables, "; ".join(notes)


def compare_from_published(A, B, names=("A", "B"),
                           metrics=("sensitivity", "specificity", "accuracy"),
                           alpha=ALPHA):
    """
    A, B : (n_pos, n_neg, sens_pct, spec_pct) per modality; percentages may be
           strings to preserve the reported precision (e.g. "84.8").
    Runs the comparison for every reconstructed table pair and reports the
    range of each comparison statistic. PPV/NPV are omitted by default because
    they inherit both ambiguities and are rarely the published comparison.
    """
    tabs_A, note_A = reconstruct_tables(*A)
    tabs_B, note_B = reconstruct_tables(*B)
    if not tabs_A or not tabs_B:
        raise ValueError(f"Reconstruction failed. {names[0]}: {note_A or 'ok'}. {names[1]}: {note_B or 'ok'}.")
    runs = [compare_modalities(tA, tB, names=names, metrics=metrics, alpha=alpha)
            for tA in tabs_A for tB in tabs_B]
    out = {"reconstructed": True, "n_table_pairs": len(runs),
           "tables": {names[0]: tabs_A, names[1]: tabs_B},
           "notes": {names[0]: note_A, names[1]: note_B}, "metrics": {}}
    for m in metrics:
        vals = {}
        for key in ("p", "GFI", "GFQ", "RQ"):
            xs = [r[m]["comparison"].get(key) for r in runs if "error" not in r[m]]
            xs = [x for x in xs if x is not None]
            vals[key] = (min(xs), max(xs)) if xs else (None, None)
        sig = {r[m]["comparison"]["significant"] for r in runs if "error" not in r[m]}
        vals["significant"] = ("yes" if sig == {True} else "no" if sig == {False} else "DEPENDS ON ROUNDING")
        out["metrics"][m] = vals
    out["first_run"] = runs[0]
    return out


def compare_counts(kA, nA, kB, nB, names=("A", "B"), metric="accuracy", alpha=ALPHA):
    """Mode 3: one metric from correct/n per modality (unpaired route)."""
    row = {
        "metric": metric,
        names[0]: {"k": kA, "n": nA, "value": kA / nA, "ci95": clopper_pearson(kA, nA, alpha)},
        names[1]: {"k": kB, "n": nB, "value": kB / nB, "ci95": clopper_pearson(kB, nB, alpha)},
        "difference_B_minus_A": kB / nB - kA / nA,
        "comparison": unpaired_triplet(kA, nA, kB, nB, alpha=alpha),
    }
    return {metric: row}


def print_published_report(out, names=("A", "B")):
    print(f"\nToolkit engine: {TOOLKIT_NAME}")
    print("RECONSTRUCTED TABLES from published n_pos, n_neg, sensitivity %, specificity %.")
    for nm in names:
        print(f"  {nm}: {len(out['tables'][nm])} candidate table(s) (TP, FN, FP, TN): "
              f"{out['tables'][nm]}" + (f"  [{out['notes'][nm]}]" if out['notes'][nm] else ""))
    print(f"  {out['n_table_pairs']} table pair(s) evaluated; ranges below are min-max over pairs.")
    print("metric\tp\tsignificant\tGFI\tGFQ\tRQ")

    def rng(t, nd=4):
        lo, hi = t
        if lo is None:
            return "NA"
        if isinstance(lo, int) and isinstance(hi, int):
            return f"{lo}" if lo == hi else f"{lo}-{hi}"
        return f"{lo:.{nd}f}" if abs(hi - lo) < 10 ** (-nd) else f"{lo:.{nd}f}-{hi:.{nd}f}"

    for m, v in out["metrics"].items():
        print(f"{m}\t{rng(v['p'])}\t{v['significant']}\t{rng(v['GFI'])}\t{rng(v['GFQ'])}\t{rng(v['RQ'])}")
    print("\nFull report for the first table pair:")
    print_report(out["first_run"], names=names)


# ---------- paired comparison: exact McNemar + PFI-M + MHQ ----------

def mcnemar_exact_p(b, c):
    if b + c == 0:
        return 1.0
    return float(binomtest(b, b + c, 0.5, alternative="two-sided").pvalue)


def mcnemar_chi2_p(b, c):
    if b + c <= 0:
        return 1.0
    return float(chi2.sf((b - c) ** 2 / (b + c), df=1))


def pfi_m(a, b, c, d, alpha=ALPHA, tol=1e-12, max_iter=300):
    """PFI-M: McNemar chi-square along the path (a+x, b-x, c-x, d+x); 4|x|/N."""
    a, b, c, d = map(float, (a, b, c, d))
    n = a + b + c + d
    if b + c == 0:
        return 1.0, True
    x_min, x_max = -min(a, d), min(b, c)
    p0 = mcnemar_chi2_p(b, c)
    sig0 = p0 < alpha

    def f(x):
        bb, cc = b - x, c - x
        if min(a + x, bb, cc, d + x) < 0:
            return float("nan")
        return mcnemar_chi2_p(bb, cc) - alpha

    lo, hi, boundary_x = (x_min, 0.0, x_min) if sig0 else (0.0, x_max, x_max)
    if abs(lo - hi) < tol:
        return max(0.0, min(1.0, 4.0 * abs(boundary_x) / n)), True
    flo, fhi = f(lo), f(hi)
    if not (flo == flo and fhi == fhi) or flo * fhi > 0:
        return max(0.0, min(1.0, 4.0 * abs(boundary_x) / n)), True
    x_star = None
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        fmid = f(mid)
        if abs(hi - lo) < tol or fmid == 0:
            x_star = mid
            break
        if flo * fmid <= 0:
            hi, fhi = mid, fmid
        else:
            lo, flo = mid, fmid
    if x_star is None:
        x_star = 0.5 * (lo + hi)
    return (max(0.0, min(1.0, 4.0 * abs(x_star) / n)),
            abs(abs(x_star) - abs(boundary_x)) < tol)


def paired_triplet(kA, nA, kB, nB, b, c, alpha=ALPHA):
    """b = correct on A / wrong on B; c = wrong on A / correct on B."""
    if nA != nB:
        raise ValueError("Paired design requires the same relevant subset for both modalities.")
    if kA - kB != b - c:
        raise ValueError("Inconsistent counts: correct_A - correct_B must equal b - c.")
    a = kA - b            # correct on both
    d = nA - a - b - c    # wrong on both
    if min(a, d) < 0:
        raise ValueError("Discordant counts exceed what the marginals allow.")
    p = mcnemar_exact_p(b, c)
    pfi, limited = pfi_m(a, b, c, d, alpha)
    mhq = 0.0 if b + c == 0 else abs(b - c) / (b + c)
    return {
        "design": "paired analysis (exact McNemar; PFI-M + MHQ)",
        "table": (a, b, c, d), "N": nA,
        "p": p, "significant": p < alpha,
        "PFI_M": pfi, "PFI_M_boundary_limited": limited,
        "MHQ": mhq,
    }


# ---------- high-level ----------

def compare_modalities(ct, pet, names=("A", "B"), paired=None,
                       metrics=("sensitivity", "specificity", "accuracy", "ppv", "npv"),
                       alpha=ALPHA):
    """
    ct, pet : (TP, FN, FP, TN) for modality A and modality B.
    paired  : optional dict {metric: (b, c)} of discordant-pair counts.
    Returns a dict keyed by metric.
    """
    paired = paired or {}
    mA, mB = metric_counts(*ct), metric_counts(*pet)
    out = {}
    for m in metrics:
        kA, nA = mA[m]
        kB, nB = mB[m]
        if nA == 0 or nB == 0:
            out[m] = {"error": "denominator zero"}
            continue
        row = {
            "metric": m,
            names[0]: {"k": kA, "n": nA, "value": kA / nA, "ci95": clopper_pearson(kA, nA, alpha)},
            names[1]: {"k": kB, "n": nB, "value": kB / nB, "ci95": clopper_pearson(kB, nB, alpha)},
            "difference_B_minus_A": kB / nB - kA / nA,
        }
        if m in paired:
            row["comparison"] = paired_triplet(kA, nA, kB, nB, *paired[m], alpha=alpha)
        else:
            row["comparison"] = unpaired_triplet(kA, nA, kB, nB, alpha=alpha)
        out[m] = row
    return out


def _fmt(x, nd=4):
    return "NA" if x is None else f"{x:.{nd}f}"


def print_report(res, names=("A", "B")):
    print(f"\nToolkit engine: {TOOLKIT_NAME}")
    print("metric\tdesign\t" + f"{names[0]}\t{names[1]}\tdiff(B-A)\tp\tfr\tnb")
    for m, row in res.items():
        if "error" in row:
            print(f"{m}\t{row['error']}")
            continue
        cmp_ = row["comparison"]
        vA, vB = row[names[0]], row[names[1]]
        if "GFQ" in cmp_:
            fr = f"GFQ={_fmt(cmp_['GFQ'])} (GFI={cmp_['GFI']}, {cmp_['fr_band']})"
            nb = f"RQ={_fmt(cmp_['RQ'])}"
        else:
            lim = " boundary-limited" if cmp_["PFI_M_boundary_limited"] else ""
            fr = f"PFI-M={_fmt(cmp_['PFI_M'])}{lim}"
            nb = f"MHQ={_fmt(cmp_['MHQ'])}"
        print(f"{m}\t{cmp_['design'].split(' (')[0]}\t"
              f"{vA['k']}/{vA['n']}={_fmt(vA['value'], 3)}\t"
              f"{vB['k']}/{vB['n']}={_fmt(vB['value'], 3)}\t"
              f"{row['difference_B_minus_A']:+.3f}\t{_fmt(cmp_['p'], 4)}\t{fr}\t{nb}")
    print()
    for m, row in res.items():
        if "error" in row:
            continue
        cmp_ = row["comparison"]
        vA, vB = row[names[0]], row[names[1]]
        print(f"{m.upper()}: {names[0]} {vA['k']}/{vA['n']} ({100*vA['value']:.1f}%, "
              f"95% CI {100*vA['ci95'][0]:.1f}-{100*vA['ci95'][1]:.1f}); "
              f"{names[1]} {vB['k']}/{vB['n']} ({100*vB['value']:.1f}%, "
              f"95% CI {100*vB['ci95'][0]:.1f}-{100*vB['ci95'][1]:.1f}).")
        print(f"  {cmp_['design']}; comparison table {cmp_['table']}, N = {cmp_['N']}; "
              f"p = {cmp_['p']:.4f} ({'significant' if cmp_['significant'] else 'nonsignificant'}).")
        if "GFI" in cmp_:
            cert = "certified exact" if cmp_["GFI_certified"] else "UPPER BOUND"
            print(f"  GFI = {cmp_['GFI']} ({cert}); GFQ = {_fmt(cmp_['GFQ'])}, {cmp_['fr_band']}; "
                  f"RQ = {_fmt(cmp_['RQ'])}.")


def main():
    print("Modality comparison: two diagnostic tests against the same reference standard.")
    nA = input("Name of modality A [A]: ").strip() or "A"
    nB = input("Name of modality B [B]: ").strip() or "B"

    print("\nInput mode: 1 = raw counts (TP, FN, FP, TN)   2 = published n_pos, n_neg, sens %, spec %"
          "   3 = correct and n per modality, one metric")
    mode = input("Choose (1, 2 or 3) [1]: ").strip() or "1"

    if mode == "3":
        metric = input("Metric name [accuracy]: ").strip() or "accuracy"
        kA = int(input(f"  {nA} correct: ").strip())
        n_A = int(input(f"  {nA} n: ").strip())
        kB = int(input(f"  {nB} correct: ").strip())
        n_B = int(input(f"  {nB} n: ").strip())
        res = compare_counts(kA, n_A, kB, n_B, names=(nA, nB), metric=metric)
        print_report(res, names=(nA, nB))
        print("\nCitation: Heston TF (2025). Fragility Metrics Toolkit [Software]. Zenodo. "
              "https://doi.org/10.5281/zenodo.17254763")
        return

    if mode == "2":
        def read_pub(name):
            print(f"\n{name}: enter the published figures (keep the paper's decimals, e.g. 84.8).")
            n_pos = int(input("  reference-positive cases (n_pos): ").strip())
            n_neg = int(input("  reference-negative cases (n_neg): ").strip())
            sens = input("  sensitivity %: ").strip()
            spec = input("  specificity %: ").strip()
            return (n_pos, n_neg, sens, spec)
        A = read_pub(nA)
        B = read_pub(nB)
        out = compare_from_published(A, B, names=(nA, nB))
        print_published_report(out, names=(nA, nB))
        print("\nCitation: Heston TF (2025). Fragility Metrics Toolkit [Software]. Zenodo. "
              "https://doi.org/10.5281/zenodo.17254763")
        return

    def read4(name):
        print(f"\n{name}: enter TP, FN, FP, TN as integers.")
        return tuple(int(input(f"  {lab}: ").strip()) for lab in ("TP", "FN", "FP", "TN"))

    ct = read4(nA)
    pet = read4(nB)
    paired = {}
    ans = input("\nDiscordant-pair counts available for any metric (same cases read by both)? [y/N]: ").strip().lower()
    if ans in ("y", "yes"):
        for m in ("sensitivity", "specificity", "accuracy", "ppv", "npv"):
            s = input(f"  {m}: b (correct {nA}/wrong {nB}), c (wrong {nA}/correct {nB}), blank to skip: ").strip()
            if s:
                b, c = (int(x) for x in s.replace(",", " ").split())
                paired[m] = (b, c)
    res = compare_modalities(ct, pet, names=(nA, nB), paired=paired)
    print_report(res, names=(nA, nB))
    print("\nCitation: Heston TF (2025). Fragility Metrics Toolkit [Software]. Zenodo. "
          "https://doi.org/10.5281/zenodo.17254763")


if __name__ == "__main__":
    main()
